# 4. Model

This notebook assesses the model, trains its final version and writes the Kaggle submission. The model is a logistic regression on the columns kept by the feature-selection strategy saved in `logs/feature_selection.json` (an L1-penalised logistic regression with C=0.1).

- **Assessment:** the same 7 temporal splits as the pre-processing and feature-selection notebooks. Inside each split, the recipe, the L1 selection, the model and the cut-off are fitted on the training rows only; the validation rows are only scored.
- **Final model:** the same steps fitted once on every training row, then used to predict every row of `test.csv`.

## 4.1 Setup

In [1]:
# Standard library
import json
import sys
import warnings
from pathlib import Path

# Arrays, tables, the model and its metrics
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

# The group's own functions, beside the feature-selection notebook
sys.path.append("../3_Feature_selection")
from preprocessing import (RANDOM_STATE, fold_matrices, load_data, paired_delta, purchase_dates,
                           target, temporal_splits)

warnings.filterwarnings("ignore", category=UserWarning)  # unknown levels in validation, as in feature selection

In [2]:
# The feature-selection strategy: the candidate columns and the L1 selector's settings
strategy = json.loads(Path("../../logs/feature_selection.json").read_text(encoding="utf-8"))
numeric = strategy["candidate_columns"]["numeric"]
categorical = strategy["candidate_columns"]["categorical"]
selector = strategy["selector"]

train_clientes = load_data()
frame = train_clientes[numeric + categorical + [target]]
time_splits = temporal_splits(frame)

# The pre-processing team's rules (1.6, 1.7.2 and 1.8), as in the feature-selection notebook
labels_as_numbers = ["is_first_purchase", "is_weekend", "since_2015_or_earlier", "purchase_year",
                     "purchase_quarter", "purchase_month", "purchase_day", "purchase_dayofweek",
                     "customer_since_year"]
recipe = {"numeric": numeric, "categorical": categorical,
          "fill": [(numeric, SimpleImputer(strategy="median")),
                   (categorical, SimpleImputer(strategy="most_frequent"))],
          "log1p": [c for c in numeric
                    if c not in labels_as_numbers and not (train_clientes[c] < 0).any()],
          "rare": {"transport_method": 50, "payment_type": 20, "city": 30, "municipality": 30,
                   "postal_area": 20}}
print(f"{len(frame):,} training rows | {len(numeric)} numeric and {len(categorical)} categorical"
      f" candidate columns | {len(time_splits)} temporal splits")

6,469 training rows | 37 numeric and 8 categorical candidate columns | 7 temporal splits


## 4.2 Assessment on the Temporal Splits

Each split trains on the past and validates on the months after it, with the 90-day gap of section 1.5.2. Inside each split, everything that learns from the data is fitted on the training rows:

1. **The recipe** (fill, `log1p`, rare levels, one-hot encoding and scaling) with `fold_matrices`.
2. **The L1 selection**, with the settings saved in the strategy.
3. **The logistic regression** on the columns L1 keeps.
4. **The cut-off:** the value with the highest F1 on the training rows. F1 is the competition metric, and when most customers come back a cut-off below 0.5 catches more of them; choosing it on the training rows keeps the validation rows out of that choice.

The validation rows are scored with F1 at that cut-off, AP and ROC-AUC (how well the probabilities rank the customers, without a cut-off), balanced accuracy, precision and recall. The training F1 sits beside the validation F1, so a wide gap would show overfitting.

In [3]:
CUTOFFS = np.round(np.arange(0.05, 0.96, 0.01), 2)


def make_selector():
    """A fresh, unfitted L1 selector with the settings saved by the feature-selection notebook."""
    return SelectFromModel(
        LogisticRegression(l1_ratio=selector["l1_ratio"], solver=selector["solver"], C=selector["C"],
                           max_iter=selector["max_iter"], random_state=selector["random_state"]),
        threshold=selector["threshold"])


def make_model():
    """A fresh, unfitted model: the logistic regression used throughout the project."""
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)


def best_cutoff(y, probabilities):
    """The cut-off with the highest F1 on these rows."""
    return CUTOFFS[np.argmax([f1_score(y, probabilities >= c) for c in CUTOFFS])]

In [4]:
rows, curves, pooled = [], [], []
dates = purchase_dates(frame)
for train_rows, validation_rows in time_splits:
    train, validation = frame.iloc[train_rows], frame.iloc[validation_rows]
    A_train, A_validation, _ = fold_matrices(train, validation, recipe)
    keep = make_selector().fit(A_train, train[target]).get_support()
    model = make_model().fit(A_train[:, keep], train[target])
    p_train = model.predict_proba(A_train[:, keep])[:, 1]
    p_validation = model.predict_proba(A_validation[:, keep])[:, 1]

    # The cut-off is chosen on the training rows only
    cutoff = best_cutoff(train[target], p_train)
    y, predicted = validation[target], (p_validation >= cutoff).astype(int)
    rows.append({"validation from": dates.iloc[validation_rows].min().date(),
                 "until": dates.iloc[validation_rows].max().date(),
                 "columns": int(keep.sum()), "cut-off": cutoff,
                 "train F1": f1_score(train[target], p_train >= cutoff),
                 "F1": f1_score(y, predicted),
                 "AP": average_precision_score(y, p_validation),
                 "ROC-AUC": roc_auc_score(y, p_validation),
                 "balanced accuracy": balanced_accuracy_score(y, predicted),
                 "precision": precision_score(y, predicted),
                 "recall": recall_score(y, predicted),
                 # The same model at the default cut-off, and the two floors of section 4.3
                 "repeat rate": y.mean(),
                 "F1 at 0.5": f1_score(y, p_validation >= 0.5),
                 "always 1": f1_score(y, np.ones(len(y), dtype=int)),
                 "professor's rule": f1_score(y, (validation["purchase_count_365d"] >= 3).astype(int))})
    curves.append(pd.DataFrame({"cut-off": CUTOFFS,
                                "train F1": [f1_score(train[target], p_train >= c) for c in CUTOFFS],
                                "validation F1": [f1_score(y, p_validation >= c) for c in CUTOFFS]}))
    pooled.append(pd.DataFrame({"actual": y.to_numpy(), "predicted": predicted}))

assessment = pd.DataFrame(rows)
measures = ["train F1", "F1", "AP", "ROC-AUC", "balanced accuracy", "precision", "recall"]
print(assessment[["validation from", "until", "columns", "cut-off"] + measures]
      .round(3).to_string(index=False))
print("\nmean:", assessment[measures].mean().round(4).to_dict())
print("SEM: ", (assessment[measures].std(ddof=1) / np.sqrt(len(assessment))).round(4).to_dict())

validation from      until  columns  cut-off  train F1    F1    AP  ROC-AUC  balanced accuracy  precision  recall
     2019-06-09 2020-03-28       22     0.37     0.769 0.741 0.845    0.771              0.679      0.704   0.782
     2020-03-28 2021-03-06       24     0.37     0.768 0.744 0.838    0.747              0.669      0.734   0.755
     2021-03-06 2021-11-13       27     0.41     0.764 0.752 0.858    0.820              0.725      0.738   0.766
     2021-11-13 2022-10-02       30     0.44     0.763 0.726 0.825    0.793              0.702      0.674   0.786
     2022-10-04 2023-07-28       33     0.41     0.763 0.741 0.823    0.794              0.719      0.690   0.800
     2023-07-29 2024-04-13       32     0.41     0.757 0.789 0.872    0.796              0.714      0.781   0.798
     2024-04-13 2025-02-15       34     0.40     0.760 0.802 0.860    0.798              0.705      0.743   0.871

mean: {'train F1': 0.7634, 'F1': 0.7565, 'AP': 0.8459, 'ROC-AUC': 0.7884, 'balanced acc

**Conclusions:**
- Over the 7 splits the model reaches **F1 0.757 ± 0.011** (mean ± SEM) at its cut-off, with AP 0.846, ROC-AUC 0.788 and balanced accuracy 0.702. The training F1 (0.763) stays close to the validation F1, so the model is not overfitting.
- F1 depends on the period: from 0.726 in the block starting in November 2021 to 0.802 in the latest block (April 2024 to February 2025), the one closest to the test period.
- The cut-off chosen on the training rows is stable, between 0.37 and 0.44, and L1 keeps between 22 and 34 columns as the training rows grow.

## 4.3 Against the Floors

F1 ignores the customers correctly left out, so a high F1 means little without a floor beside it. The model is compared, split by split, with:

- **always 1:** predicting a repeat purchase for every customer, the floor of the course (`DummyClassifier` with the majority class);
- **the professor's rule:** the baseline in `sample_submission.csv`, which predicts a repeat purchase when `purchase_count_365d >= 3` (the first line below checks it on the test rows); on the public leaderboard it scores 0.7794;
- **the same model at the default cut-off of 0.5.**

In [5]:
# The professor's baseline is a rule: every row of sample_submission.csv follows it
sample = pd.read_csv("../../data/sample_submission.csv")
test_raw = pd.read_csv("../../data/test.csv").merge(sample, on="ID")
follows = ((test_raw["purchase_count_365d"] >= 3).astype(int) == test_raw[target]).mean()
print(f"sample_submission rows that follow purchase_count_365d >= 3: {follows:.0%}\n")

compared = ["always 1", "professor's rule", "F1 at 0.5", "F1"]
print(assessment[["validation from", "repeat rate"] + compared].round(3).to_string(index=False))
print("\nmean:", assessment[compared].mean().round(4).to_dict())
for name in ("professor's rule", "F1 at 0.5"):
    r = paired_delta(assessment["F1"], assessment[name])
    print(f"model at its cut-off minus {name}: dF1 {r['delta']:+.4f} +/- {r['SEM']:.4f},"
          f" better on {r['wins']} -> {r['verdict']}")

sample_submission rows that follow purchase_count_365d >= 3: 100%

validation from  repeat rate  always 1  professor's rule  F1 at 0.5    F1
     2019-06-09        0.563     0.720             0.733      0.729 0.741
     2020-03-28        0.604     0.753             0.745      0.715 0.744
     2021-03-06        0.537     0.699             0.740      0.751 0.752
     2021-11-13        0.502     0.668             0.719      0.715 0.726
     2022-10-04        0.502     0.668             0.728      0.734 0.741
     2023-07-29        0.622     0.767             0.771      0.763 0.789
     2024-04-13        0.605     0.754             0.787      0.774 0.802

mean: {'always 1': 0.7186, "professor's rule": 0.7461, 'F1 at 0.5': 0.7402, 'F1': 0.7565}
model at its cut-off minus professor's rule: dF1 +0.0104 +/- 0.0023, better on 6/7 -> better
model at its cut-off minus F1 at 0.5: dF1 +0.0163 +/- 0.0043, better on 7/7 -> better


**Conclusions:**
- The professor's baseline is exactly the rule `purchase_count_365d >= 3`: every row of `sample_submission.csv` follows it.
- At its cut-off, the model beats both floors: always 1 (0.719) and the professor's rule (0.746), by **+0.010 ± 0.002** F1, ahead on 6 of the 7 splits.
- At the default cut-off of 0.5 it would not (0.740, behind the rule). Choosing the cut-off on the training rows is worth **+0.016 ± 0.004** F1, ahead on all 7 splits.
- When many customers come back, predicting a repeat purchase for everyone already scores a high F1: in the blocks starting in March 2020 and July 2023 (repeat rates of 0.60 and 0.62), always 1 scores above the model at 0.5.

## 4.4 Reading the Errors

The confusion matrix gathers the predictions of the 7 validation blocks. The table beside it shows the mean F1 of each cut-off on the training rows, where the cut-off is chosen, and on the validation rows, where it is checked.

In [6]:
both = pd.concat(pooled, ignore_index=True)
tn, fp, fn, tp = confusion_matrix(both["actual"], both["predicted"]).ravel()
print(f"{len(both):,} validation purchases: {tp} repeat purchases caught, {fn} missed,"
      f" {fp} false alarms, {tn} correctly left out\n")

curve = pd.concat(curves).groupby("cut-off").mean()
print(curve.loc[[0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]].round(3).to_string())
print(f"\nbest mean F1: at {curve['train F1'].idxmax()} on the training rows,"
      f" at {curve['validation F1'].idxmax()} on the validation rows")

4,116 validation purchases: 1839 repeat purchases caught, 475 missed, 699 false alarms, 1103 correctly left out

         train F1  validation F1
cut-off                         
0.30        0.745          0.747
0.35        0.758          0.756
0.40        0.761          0.756
0.45        0.758          0.749
0.50        0.745          0.740
0.55        0.728          0.724
0.60        0.704          0.697

best mean F1: at 0.41 on the training rows, at 0.39 on the validation rows


**Conclusions:**
- Of the 2,314 repeat purchases in the validation blocks, the model catches 1,839 (recall 0.79) and misses 475. Of its 2,538 positive predictions, 699 are false alarms (precision 0.72).
- The best mean cut-off on the training rows (0.41) is almost the best on the validation rows (0.39), and every value between 0.35 and 0.45 scores about the same on the validation rows (0.749 to 0.756): the choice carries over to new rows and does not hinge on a precise value. Above 0.5, F1 drops quickly.

## 4.5 Final Model and Kaggle Submission

The final model repeats the steps of each split once, on every training row. The feature-selection tables already hold them: `feature_selection_l1.csv` carries the training rows prepared by the recipe and reduced to the columns L1 kept, both fitted on every training row, and `feature_selection_l1_test.csv` carries the test rows prepared with those same fits. The cut-off is chosen on the training rows, as in each split, and the submission keeps `ID` and writes 0/1 labels. Each new submission gets the next version number in its file name.

In [7]:
train_table = pd.read_csv("../../data/featureselection/feature_selection_l1.csv")
test_table = pd.read_csv("../../data/featureselection/feature_selection_l1_test.csv")
features = [c for c in train_table.columns if c not in ("ID", "customer_id", target)]
# The test table must carry the same columns, in the same order
assert list(test_table.columns) == list(train_table.columns.drop(target))

final_model = make_model().fit(train_table[features], train_table[target])
final_cutoff = best_cutoff(train_table[target], final_model.predict_proba(train_table[features])[:, 1])
test_probabilities = final_model.predict_proba(test_table[features])[:, 1]
submission = pd.DataFrame({"ID": test_table["ID"],
                           target: (test_probabilities >= final_cutoff).astype(int)})

# The same rows and IDs as sample_submission.csv, with labels rather than probabilities
assert list(submission["ID"]) == list(sample["ID"]) and set(submission[target]) <= {0, 1}

submission_file = Path("../../data/submissions/ML_Group_18_Version01.csv")
submission_file.parent.mkdir(exist_ok=True)
submission.to_csv(submission_file, index=False)
print(f"{len(features)} features | cut-off {final_cutoff} | {submission[target].mean():.1%} of the"
      f" test rows predicted to come back | saved to {submission_file.as_posix()}")

39 features | cut-off 0.41 | 71.4% of the test rows predicted to come back | saved to ../../data/submissions/ML_Group_18_Version01.csv


## 4.6 Conclusions

- **Model:** a logistic regression on the 39 columns the L1 strategy keeps, with the cut-off chosen on the training rows (0.41 on every training row).
- **Assessed performance:** F1 **0.757 ± 0.011** over the 7 temporal splits, from 0.726 to 0.802 depending on the period, against 0.746 for the professor's rule and 0.719 for always 1. This is the performance the Kaggle result should be read against.
- **Submission:** `ML_Group_18_Version01.csv` holds the 986 test IDs.
- **Limits:** the recipe, the feature-selection strategy and the cut-off rule were all decided on these same 7 splits, so the estimate is slightly optimistic. Only one model is assessed here; the final project compares at least five, with their hyperparameters tuned inside the temporal splits.